# 06 오답 분석 및 최종 모델 선택

05번 결과를 바탕으로 최종 모델을 선택하고, 선택 모델 (Metadata only + MLP)의 오답 데이터를 분석한다.

참고로 06번은 별점 정제 단계가 아니다. 07번에서 저장된 선택 모델을 불러와 별점 정제를 진행한다.

## 1. 라이브러리 로드

저장된 모델을 불러오고, test split 기준 성능과 오답 데이터를 다시 계산한다.


In [1]:
import os

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split


## 2. Load Validation/Test Data and Raw Review Rows

Reproduce the same split as notebook 05 and connect validation/test rows back to the original review rows.

- validation: final model selection
- test: final reference performance and error analysis

This mapping allows FP/FN examples to include review text and metadata.

In [2]:
SEED = 42
LABEL_COL = 'label'
TEXT_COL = 'cleaned_review_text'
META_COLS = ['text_length', 'emoji_count', 'photo_count']
RAW_META_COLS = ['text_length', 'emoji_count', 'photo_count']

raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')

emb_cols = [f'kcbert_{i}' for i in range(768)]
meta_cols = META_COLS
hybrid_cols = emb_cols + meta_cols

X_all = raw_df[hybrid_cols].copy()
y_all = raw_df[LABEL_COL].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all,
    y_all,
    test_size=0.3,
    random_state=SEED,
    stratify=y_all,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=y_temp,
)

val_review_df = raw_df.loc[X_val.index].copy().reset_index().rename(columns={'index': 'raw_index'})
test_review_df = raw_df.loc[X_test.index].copy().reset_index().rename(columns={'index': 'raw_index'})

text_val = val_review_df[TEXT_COL].fillna('').astype(str)
text_test = test_review_df[TEXT_COL].fillna('').astype(str)

print('validation feature shape:', X_val.shape)
print('test feature shape:', X_test.shape)
print('validation review shape:', val_review_df.shape)
print('test review shape:', test_review_df.shape)
print('validation label ??:', y_val.value_counts().sort_index().to_dict())
print('test label ??:', y_test.value_counts().sort_index().to_dict())
print('excluded feature:', ['rating'])

validation feature shape: (1326, 771)
test feature shape: (1327, 771)
validation review shape: (1326, 784)
test review shape: (1327, 784)
validation label ??: {0: 854, 1: 472}
test label ??: {0: 854, 1: 473}
excluded feature: ['rating']


C:\Users\ICT\AppData\Local\Temp\ipykernel_3404\3898116630.py:7: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')


## 3. 저장된 모델 bundle 로드

05번에서 저장한 baseline/ablation 모델과 04번에서 저장한 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)을 모두 불러온다.


In [3]:
model_specs = [
    {
        'model': 'baseline_tfidf_random_forest',
        'feature_set': 'cleaned_text_tfidf',
        'path': 'outputs/baseline_tfidf_random_forest_model.joblib',
        'input_type': 'text',
    },
    {
        'model': 'ablation_mlp_kcbert_pca_only',
        'feature_set': 'kcbert_pca_only',
        'path': 'outputs/ablation_mlp_kcbert_pca_only_model.joblib',
        'input_type': 'tabular',
    },
    {
        'model': 'ablation_mlp_metadata_only',
        'feature_set': 'metadata_only',
        'path': 'outputs/ablation_mlp_metadata_only_model.joblib',
        'input_type': 'tabular',
    },
    {
        'model': 'proposed_hybrid_mlp_04',
        'feature_set': 'kcbert_pca+metadata',
        'path': 'outputs/proposed_mlp_final_model.joblib',
        'input_type': 'tabular',
    },
]

missing_paths = [spec['path'] for spec in model_specs if not os.path.exists(spec['path'])]
if missing_paths:
    raise FileNotFoundError(f'모델 파일이 없습니다: {missing_paths}')

model_bundles = {spec['model']: joblib.load(spec['path']) for spec in model_specs}

loaded_models = pd.DataFrame([
    {'model': spec['model'], 'feature_set': spec['feature_set'], 'path': spec['path']}
    for spec in model_specs
])
display(loaded_models)


,model,feature_set,path
0,baseline_tfidf_random_forest,cleaned_text_tfidf,outputs/baseline_tfidf_random_forest_model.joblib
1,ablation_mlp_kcbert_pca_only,kcbert_pca_only,outputs/ablation_mlp_kcbert_pca_only_model.joblib
2,ablation_mlp_metadata_only,metadata_only,outputs/ablation_mlp_metadata_only_model.joblib
3,proposed_hybrid_mlp_04,kcbert_pca+metadata,outputs/proposed_mlp_final_model.joblib


## 4. Compare Models on Validation and Test

Apply all saved models to the same validation/test splits.

- Validation metrics are used for final model selection.
- Test metrics are reference metrics after selection.
- Each model uses the tuned threshold stored in its model bundle.

In [4]:
def predict_prob(spec, bundle, split):
    model = bundle['model']

    if split == 'validation':
        text_data = text_val
        X_data = X_val
    elif split == 'test':
        text_data = text_test
        X_data = X_test
    else:
        raise ValueError(f'Unsupported split: {split}')

    if spec['input_type'] == 'text':
        return model.predict_proba(text_data)[:, 1]

    feature_cols = bundle['feature_cols']
    return model.predict_proba(X_data[feature_cols])[:, 1]


def metric_row(spec, bundle, split, y_true, prob):
    threshold = float(bundle.get('best_threshold', bundle.get('default_threshold', 0.5)))
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    return {
        'model': spec['model'],
        'feature_set': spec['feature_set'],
        'split': split,
        'threshold': threshold,
        'f1': float(f1_score(y_true, pred)),
        'pr_auc': float(average_precision_score(y_true, prob)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, prob)),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
    }


model_probabilities = {'validation': {}, 'test': {}}
rows = []
for spec in model_specs:
    bundle = model_bundles[spec['model']]

    val_prob = predict_prob(spec, bundle, split='validation')
    test_prob = predict_prob(spec, bundle, split='test')

    model_probabilities['validation'][spec['model']] = val_prob
    model_probabilities['test'][spec['model']] = test_prob

    rows.append(metric_row(spec, bundle, 'validation', y_val, val_prob))
    rows.append(metric_row(spec, bundle, 'test', y_test, test_prob))

comparison_all = pd.DataFrame(rows)
validation_comparison = (
    comparison_all[comparison_all['split'] == 'validation']
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
validation_comparison.insert(0, 'rank', validation_comparison.index + 1)

test_comparison = (
    comparison_all[comparison_all['split'] == 'test']
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
test_comparison.insert(0, 'rank', test_comparison.index + 1)

print('Validation ranking for model selection')
display(validation_comparison.round({
    'threshold': 4,
    'f1': 4,
    'pr_auc': 4,
    'precision': 4,
    'recall': 4,
    'roc_auc': 4,
}))

print('Test ranking for reference')
display(test_comparison.round({
    'threshold': 4,
    'f1': 4,
    'pr_auc': 4,
    'precision': 4,
    'recall': 4,
    'roc_auc': 4,
}))

Validation ranking for model selection


,rank,model,feature_set,split,threshold,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,1,proposed_hybrid_mlp_04,kcbert_pca+metadata,validation,0.5038,0.4444,0.4070,0.4116,0.4831,0.5561,528,326,244,228
1,2,ablation_mlp_metadata_only,metadata_only,validation,0.5009,0.4257,0.3809,0.3932,0.4640,0.5533,516,338,253,219
2,3,ablation_mlp_kcbert_pca_only,kcbert_pca_only,validation,0.5067,0.4068,0.3909,0.3877,0.4280,0.5358,535,319,270,202
3,4,baseline_tfidf_random_forest,cleaned_text_tfidf,validation,0.5016,0.2613,0.4015,0.4485,0.1843,0.5482,747,107,385,87


Test ranking for reference


,rank,model,feature_set,split,threshold,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,1,ablation_mlp_metadata_only,metadata_only,test,0.5009,0.4615,0.4069,0.4148,0.5201,0.5558,507,347,227,246
1,2,proposed_hybrid_mlp_04,kcbert_pca+metadata,test,0.5038,0.4271,0.4423,0.4152,0.4397,0.5784,561,293,265,208
2,3,ablation_mlp_kcbert_pca_only,kcbert_pca_only,test,0.5067,0.3974,0.4307,0.4108,0.3848,0.5810,593,261,291,182
3,4,baseline_tfidf_random_forest,cleaned_text_tfidf,test,0.5016,0.2927,0.4381,0.5246,0.2030,0.5837,767,87,377,96


## 5. Final Model Selection

Select the final model by validation F1 for event reviews. Use PR-AUC and the precision/recall balance as secondary context.

Record test performance for the selected model, but do not switch models based on test results.

In [5]:
selected_row = validation_comparison.iloc[0].copy()
selected_model_name = selected_row['model']
selected_spec = next(spec for spec in model_specs if spec['model'] == selected_model_name)
selected_bundle = model_bundles[selected_model_name]
selected_threshold = float(selected_bundle.get('best_threshold', selected_bundle.get('default_threshold', 0.5)))

selected_test_row = test_comparison[test_comparison['model'] == selected_model_name].iloc[0].copy()

selection_reason = (
    'Selected because it has the highest event-class F1 on validation with the stored tuned threshold. '
    'Test data is used only for final reference metrics and error analysis.'
)

selected_display = pd.DataFrame([{
    'selected_model': selected_model_name,
    'feature_set': selected_row['feature_set'],
    'threshold': selected_threshold,
    'validation_f1': selected_row['f1'],
    'validation_pr_auc': selected_row['pr_auc'],
    'validation_precision': selected_row['precision'],
    'validation_recall': selected_row['recall'],
    'test_f1_reference': selected_test_row['f1'],
    'test_pr_auc_reference': selected_test_row['pr_auc'],
    'test_precision_reference': selected_test_row['precision'],
    'test_recall_reference': selected_test_row['recall'],
    'test_fp_reference': int(selected_test_row['fp']),
    'test_fn_reference': int(selected_test_row['fn']),
    'selection_reason': selection_reason,
}]).round({
    'threshold': 4,
    'validation_f1': 4,
    'validation_pr_auc': 4,
    'validation_precision': 4,
    'validation_recall': 4,
    'test_f1_reference': 4,
    'test_pr_auc_reference': 4,
    'test_precision_reference': 4,
    'test_recall_reference': 4,
})

display(selected_display)

,selected_model,feature_set,threshold,validation_f1,validation_pr_auc,validation_precision,validation_recall,test_f1_reference,test_pr_auc_reference,test_precision_reference,test_recall_reference,test_fp_reference,test_fn_reference,selection_reason
0,proposed_hybrid_mlp_04,kcbert_pca+metadata,0.5038,0.4444,0.407,0.4116,0.4831,0.4271,0.4423,0.4152,0.4397,293,265,Selected because it has the highest event-clas...


## 6. Extract FP/FN Errors for the Selected Model

On the test split, divide selected-model errors into:

- FP: actual normal review (0), predicted event review (1)
- FN: actual event review (1), predicted normal review (0)

This is post-selection diagnosis because test data was not used to choose the model.

In [6]:
selected_prob = model_probabilities['test'][selected_model_name]
selected_pred = (selected_prob >= selected_threshold).astype(int)

error_df = test_review_df.copy()
error_df['actual_label'] = y_test.to_numpy()
error_df['pred_label'] = selected_pred
error_df['event_prob'] = selected_prob
error_df['error_type'] = np.select(
    [
        (error_df['actual_label'] == 0) & (error_df['pred_label'] == 1),
        (error_df['actual_label'] == 1) & (error_df['pred_label'] == 0),
    ],
    ['FP_normal_pred_event', 'FN_event_pred_normal'],
    default='correct',
)

summary_cols = ['photo_count', 'text_length', 'emoji_count']
error_summary = (
    error_df.groupby('error_type')
    .agg(
        count=('error_type', 'size'),
        mean_event_prob=('event_prob', 'mean'),
        mean_photo_count=('photo_count', 'mean'),
        mean_text_length=('text_length', 'mean'),
        mean_emoji_count=('emoji_count', 'mean'),
    )
    .reset_index()
    .sort_values('count', ascending=False)
    .round(4)
)

display(error_summary)

,error_type,count,mean_event_prob,mean_photo_count,mean_text_length,mean_emoji_count
2,correct,769,0.3531,0.9038,51.9012,0.6021
1,FP_normal_pred_event,293,0.7150,1.0171,54.2457,0.4846
0,FN_event_pred_normal,265,0.2327,0.8755,53.0453,0.7170


## 7. 오답 샘플 출력

FP와 FN 샘플을 각각 확인한다. 여기서 은어, 신조어, 우회 표현, 짧은 리뷰, 사진 수, 별점 패턴을 같이 본다.


In [7]:
review_cols = [
    'raw_index',
    'store_name',
    'rating',
    'review_text',
    'cleaned_review_text',
    'photo_count',
    'text_length',
    'emoji_count',
    'actual_label',
    'pred_label',
    'event_prob',
    'error_type',
]
review_cols = [col for col in review_cols if col in error_df.columns]

fp_samples = (
    error_df[error_df['error_type'] == 'FP_일반을_이벤트로_오탐']
    .sort_values('event_prob', ascending=False)
    [review_cols]
    .head(15)
)

fn_samples = (
    error_df[error_df['error_type'] == 'FN_이벤트를_일반으로_미탐']
    .sort_values('event_prob', ascending=True)
    [review_cols]
    .head(15)
)

print('FP 샘플: 일반 리뷰인데 이벤트 리뷰로 예측한 사례')
display(fp_samples)

print('FN 샘플: 이벤트 리뷰인데 일반 리뷰로 놓친 사례')
display(fn_samples)


FP 샘플: 일반 리뷰인데 이벤트 리뷰로 예측한 사례


,raw_index,store_name,rating,review_text,cleaned_review_text,photo_count,text_length,emoji_count,actual_label,pred_label,event_prob,error_type


FN 샘플: 이벤트 리뷰인데 일반 리뷰로 놓친 사례


,raw_index,store_name,rating,review_text,cleaned_review_text,photo_count,text_length,emoji_count,actual_label,pred_label,event_prob,error_type


## 8. Metadata Importance and Behavior Pattern Check

If the selected model uses tabular metadata, compute permutation importance on the test split as post-selection diagnosis.

A larger F1 drop means the metadata column is more influential for the selected model.

In [8]:
def permutation_importance_for_tabular_model(model, X, y_true, threshold, columns, repeats=10, random_state=SEED):
    rng = np.random.default_rng(random_state)
    base_prob = model.predict_proba(X)[:, 1]
    base_pred = (base_prob >= threshold).astype(int)
    base_f1 = f1_score(y_true, base_pred)

    rows = []
    for col in columns:
        drops = []
        for _ in range(repeats):
            X_perm = X.copy()
            shuffled = X_perm[col].to_numpy().copy()
            rng.shuffle(shuffled)
            X_perm[col] = shuffled
            perm_prob = model.predict_proba(X_perm)[:, 1]
            perm_pred = (perm_prob >= threshold).astype(int)
            drops.append(base_f1 - f1_score(y_true, perm_pred))

        rows.append({
            'metadata': col,
            'base_f1': float(base_f1),
            'mean_f1_drop': float(np.mean(drops)),
            'std_f1_drop': float(np.std(drops)),
            'repeats': repeats,
        })
    return pd.DataFrame(rows).sort_values('mean_f1_drop', ascending=False).reset_index(drop=True)


selected_feature_cols = selected_bundle.get('feature_cols')
if selected_spec['input_type'] == 'tabular' and selected_feature_cols is not None:
    importance_cols = [col for col in meta_cols if col in selected_feature_cols]
else:
    importance_cols = []

if importance_cols:
    selected_importance = permutation_importance_for_tabular_model(
        selected_bundle['model'],
        X_test[selected_feature_cols],
        y_test,
        selected_threshold,
        importance_cols,
        repeats=10,
    )
    selected_importance_display = selected_importance.copy()
    selected_importance_display.insert(0, 'rank', selected_importance_display.index + 1)
    selected_importance_display = selected_importance_display.rename(columns={
        'metadata': 'metadata',
        'base_f1': 'base_f1',
        'mean_f1_drop': 'mean_f1_drop',
        'std_f1_drop': 'std_f1_drop',
        'repeats': 'repeats',
    }).round({
        'base_f1': 4,
        'mean_f1_drop': 4,
        'std_f1_drop': 4,
    })
    top_metadata = selected_importance.iloc[0]
    print(
        f"Most influential metadata for the selected model: {top_metadata['metadata']} "
        f"(mean F1 drop: {top_metadata['mean_f1_drop']:.4f})"
    )
    display(selected_importance_display)
else:
    selected_importance = pd.DataFrame()
    print('Permutation importance was skipped because the selected model does not directly use metadata columns.')

behavior_summary = (
    error_df.groupby('actual_label')
    .agg(
        count=('actual_label', 'size'),
        mean_photo_count=('photo_count', 'mean'),
        mean_text_length=('text_length', 'mean'),
        mean_emoji_count=('emoji_count', 'mean'),
    )
    .reset_index()
    .replace({'actual_label': {0: 'normal_review_0', 1: 'event_review_1'}})
    .round(4)
)

print('Metadata averages by actual label')
display(behavior_summary)

Most influential metadata for the selected model: text_length (mean F1 drop: 0.0155)


,rank,metadata,base_f1,mean_f1_drop,std_f1_drop,repeats
0,1,text_length,0.4271,0.0155,0.0070,10
1,2,photo_count,0.4271,0.0061,0.0080,10
2,3,emoji_count,0.4271,0.0027,0.0048,10


Metadata averages by actual label


,actual_label,count,mean_photo_count,mean_text_length,mean_emoji_count
0,normal_review_0,854,0.9016,54.9871,0.6007
1,event_review_1,473,0.9619,48.4228,0.5962


## 9. 최종 결과 해석

### 9-1. 핵심 결론

이벤트 리뷰에 대한 validation F1을 기준으로 최종 모델을 선택한다. test 데이터는 선택 이후에만 사용한다.

이는 test 성능을 직접 기준으로 모델을 선택하는 것보다 더 엄격한 기준이다.

### 9-2. 성능 비교 해석

- validation에서 최고 성능을 보인 모델을 최종 후보로 선택한다.
- 선택된 모델의 test 성능을 따로 보고한다.
- 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)이 최종 선택되지 않은 경우, 현재 하이브리드 융합 전략의 한계로 직접 설명한다.

### 9-3. 메타데이터 영향도 해석

선택된 모델에 대해 Permutation 중요도를 해석한다. `rating`은 프로젝트에서 이후에 정제될 값이므로 제외된다.

### 9-4. 오답 데이터 해석

FP (False Positive)는 실제로는 일반 리뷰이지만 모델이 이벤트 리뷰로 잘못 판단한 경우다. 이 경우 일반 리뷰임에도 행동 패턴이 이벤트 리뷰와 유사해서 모델이 이벤트 리뷰로 오탐했을 가능성이 크다.

FN (False Negative)은 실제 이벤트 리뷰이지만 모델이 일반 리뷰로 놓친 경우다. 이 경우 메타데이터 신호가 약하거나 이벤트 여부가 텍스트를 통해서만 드러나는 경우일 가능성이 있다.

### 9-5. 향후 개선 방향

가능한 개선 방식으로는 KcBERT fine-tuning, 문장 임베딩 모델 활용, 텍스트 특징 선택, 텍스트와 메타데이터 간 가중 융합이 있다.

## 10. 선택 모델 저장

validation에서 선택된 모델을 `outputs/final_selected_model.joblib`에 저장한다.

07번에서 이 파일을 로드하여 모든 리뷰의 이벤트 확률을 예측해야 한다. 번들에는 선택된 모델의 validation 평가지표와 test 참고 평가지표도 저장된다.

In [9]:
final_selected_bundle = {
    'model': selected_bundle['model'],
    'model_name': selected_model_name,
    'feature_set': selected_row['feature_set'],
    'input_type': selected_spec['input_type'],
    'feature_cols': selected_bundle.get('feature_cols'),
    'text_col': selected_bundle.get('text_col'),
    'target_col': selected_bundle.get('target_col', LABEL_COL),
    'default_threshold': float(selected_bundle.get('default_threshold', 0.5)),
    'best_threshold': selected_threshold,
    'selection_rule': 'highest F1 on validation set with stored tuned threshold, PR-AUC as secondary criterion',
    'selection_reason': selection_reason,
    'validation_selected_metrics': selected_row.to_dict(),
    'test_reference_metrics': selected_test_row.to_dict(),
    'validation_model_comparison': validation_comparison.to_dict(orient='records'),
    'test_model_comparison': test_comparison.to_dict(orient='records'),
    'metadata_importance': selected_importance.to_dict(orient='records'),
    'excluded_features': ['rating'],
    'next_step': '07_rating_refinement should load this bundle and apply it to all review rows.',
}

joblib.dump(final_selected_bundle, 'outputs/final_selected_model.joblib')

final_selected_display = {k: v for k, v in final_selected_bundle.items() if k != 'model'}
print('?? ??: outputs/final_selected_model.joblib')
display(pd.DataFrame([final_selected_display]))

?? ??: outputs/final_selected_model.joblib


,model_name,feature_set,input_type,feature_cols,text_col,target_col,default_threshold,best_threshold,selection_rule,selection_reason,validation_selected_metrics,test_reference_metrics,validation_model_comparison,test_model_comparison,metadata_importance,excluded_features,next_step
0,proposed_hybrid_mlp_04,kcbert_pca+metadata,tabular,"[kcbert_0, kcbert_1, kcbert_2, kcbert_3, kcber...",None,label,0.5,0.503774,highest F1 on validation set with stored tuned...,Selected because it has the highest event-clas...,"{'rank': 1, 'model': 'proposed_hybrid_mlp_04',...","{'rank': 2, 'model': 'proposed_hybrid_mlp_04',...","[{'rank': 1, 'model': 'proposed_hybrid_mlp_04'...","[{'rank': 1, 'model': 'ablation_mlp_metadata_o...","[{'metadata': 'text_length', 'base_f1': 0.4271...",[rating],07_rating_refinement should load this bundle a...


## 11. 07번 별점 정제와의 연결

07번은 `outputs/final_selected_model.joblib`을 로드하여 모든 행의 이벤트 리뷰 확률을 예측하고, 정제 전후 별점을 비교한다.

선택된 모델에 False Positive가 많은 경우, 이진 제거보다 확률 기반 가중 평균 방식을 선호한다.